In [1]:
!pip install fairlearn

In [10]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    equal_opportunity_difference,
    equalized_odds_difference,
    selection_rate
)

df = pd.read_csv("hospital_readmissions.csv")

df['readmitted'] = df['readmitted'].map({'NO': 0, 'YES': 1, 'No': 0, 'Yes': 1, 'no': 0, 'yes': 1})

age_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35,
    '[40-50)': 45, '[50-60)': 55, '[60-70)': 65,
    '[70-80)': 75, '[80-90)': 85, '[90-100)': 95
}
df['age'] = df['age'].map(age_map)

df = df.dropna(subset=['readmitted'])

from sklearn.preprocessing import LabelEncoder
categorical_cols = df.select_dtypes(include='object').columns
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

X = df.drop(columns=['readmitted'])
y = df['readmitted']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("\n✅ Classification Report:\n", classification_report(y_test, y_pred))

sensitive_feature = X_test['age']

metric_frame = MetricFrame(
    metrics={"accuracy": accuracy_score},
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sensitive_feature
)

dp_diff = demographic_parity_difference(y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_feature)

eo_diff = equal_opportunity_difference(y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_feature)

eod_diff = equalized_odds_difference(y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_feature)

selection_rates = MetricFrame(
    metrics=selection_rate,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sensitive_feature
)
disparate_impact = selection_rates.ratio()

print("\n🎯 Accuracy by Age Group:\n", metric_frame.by_group)
print("\n📏 Demographic Parity Difference:", dp_diff)
print("📏 Equal Opportunity Difference:", eo_diff)
print("📏 Equalized Odds Difference:", eod_diff)
print("📏 Disparate Impact Ratio:", disparate_impact)


✅ Accuracy: 0.6078

✅ Classification Report:
               precision    recall  f1-score   support

           0       0.61      0.71      0.66      2658
           1       0.60      0.49      0.54      2342

    accuracy                           0.61      5000
   macro avg       0.61      0.60      0.60      5000
weighted avg       0.61      0.61      0.60      5000


🎯 Accuracy by Age Group:
      accuracy
age          
45   0.654135
55   0.632272
65   0.624685
75   0.591882
85   0.568027
95   0.527132

📏 Demographic Parity Difference: 0.17587573585125604
📏 Equal Opportunity Difference: 0.09489513659533921
📏 Equalized Odds Difference: 0.2737400530503979
📏 Disparate Impact Ratio: 0.6280660668063601


Demographic Parity Difference:
This value tells us whether the model is treating all age groups equally when predicting readmission. In our case, the difference is around 0.1759, which means there’s a noticeable gap (about 17.6%) in how often people from different age groups are predicted as being readmitted. Ideally, this number should be close to zero. A higher number means the model is favoring or ignoring certain age groups more than others, which is unfair.

Equal Opportunity Difference:
This metric checks if the model is equally good at correctly predicting readmission for all age groups. We got a value of about 0.0949, or 9.5% difference, which shows that the model performs better for some age groups compared to others when it comes to identifying actual readmissions. For example, it might be better at catching readmissions in younger patients than older ones. Ideally, we want this number to be low so that no group is left behind in getting accurate predictions.

Equalized Odds Difference:
This one checks if the model is making similar types of mistakes for all age groups — both when it predicts correctly and when it gets things wrong. Our value is 0.2737, which is quite high. This means the model is making different kinds of errors depending on the person’s age. For example, it might wrongly say someone won’t be readmitted when they actually will — more often in one age group than another. This shows inconsistent behavior and unfair treatment between groups.

Disparate Impact Ratio:
This ratio compares how often different age groups are being predicted as readmitted. The value we got is 0.628, which means one group is only being predicted for readmission about 62.8% as often as the best-treated group. In fairness terms, this is bad — it should be at least 0.8 (the “four-fifths rule”). This shows that the model may be discriminating against some age groups, possibly older people, by not predicting their readmission enough.

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    equal_opportunity_difference,
    equalized_odds_difference,
    selection_rate
)

df = pd.read_csv("diabetic_data.csv")

df = df.drop(columns=["encounter_id", "patient_nbr"], errors="ignore")

df = df[df['readmitted'].isin(['NO', '<30', '>30'])]

df['readmitted_binary'] = df['readmitted'].apply(lambda x: 1 if x == '<30' else 0)

df.replace('?', 'Unknown', inplace=True)

categorical_cols = df.select_dtypes(include='object').columns
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

X = df.drop(columns=['readmitted', 'readmitted_binary'])
y = df['readmitted_binary']

\X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("\n✅ Classification Report:\n", classification_report(y_test, y_pred))

sensitive_feature = X_test['race']

metric_frame = MetricFrame(
    metrics={"accuracy": accuracy_score},
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sensitive_feature
)

dp_diff = demographic_parity_difference(y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_feature)

eo_diff = equal_opportunity_difference(y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_feature)

eod_diff = equalized_odds_difference(y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_feature)

selection_rates = MetricFrame(
    metrics=selection_rate,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sensitive_feature
)
disparate_impact = selection_rates.ratio()

print("\n🎯 Accuracy by Race Group:\n", metric_frame.by_group)
print("\n📏 Demographic Parity Difference:", dp_diff)
print("📏 Equal Opportunity Difference:", eo_diff)
print("📏 Equalized Odds Difference:", eod_diff)
print("📏 Disparate Impact Ratio:", disparate_impact)


✅ Accuracy: 0.8868035766925421

✅ Classification Report:
               precision    recall  f1-score   support

           0       0.89      1.00      0.94     18069
           1       0.35      0.01      0.02      2285

    accuracy                           0.89     20354
   macro avg       0.62      0.50      0.48     20354
weighted avg       0.83      0.89      0.84     20354


🎯 Accuracy by Race Group:
       accuracy
race          
0     0.885457
1     0.903226
2     0.885802
3     0.891473
4     0.886121
5     0.924276

📏 Demographic Parity Difference: 0.008064516129032258
📏 Equal Opportunity Difference: 0.01099537037037037
📏 Equalized Odds Difference: 0.01099537037037037
📏 Disparate Impact Ratio: 0.0


Demographic Parity Difference:
The demographic parity difference is around 0.008, which is very close to zero. This means the model is giving positive predictions (i.e., saying a patient will be readmitted within 30 days) at almost the same rate for all race groups. That's a good sign, showing the model is treating all races fairly in terms of how often it gives out positive results.

Equal Opportunity Difference:
The equal opportunity difference is about 0.011, also a very small value. This tells us the model is almost equally good at identifying actual readmissions for all race groups. So, if someone truly gets readmitted, the model gives each race group a similar chance of correctly catching it. Again, this means the model is not favoring or ignoring any group when it comes to true readmission cases.

Equalized Odds Difference:
The equalized odds difference is also very low (around 0.011), which shows that the model is making similar kinds of errors and correct predictions across all race groups. This is important because it means the model is being consistent, and one group is not getting more wrong or right predictions than another — a sign of fairness in both directions (true positives and false positives).

Disparate Impact Ratio:
The disparate impact ratio is 0.0, which is unexpected and a bit concerning. This usually means that at least one race group is not receiving any positive predictions at all, which could be due to a data imbalance or the model being overly cautious. Ideally, this number should be close to 1.0, meaning all groups get a fair share of positive predictions. A value of 0 might suggest an issue in either the dataset or model behavior that needs to be looked at more carefully.

In [1]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
import torch.optim as optim
from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    equal_opportunity_difference,
    equalized_odds_difference,
    selection_rate
)

df = pd.read_csv("diabetic_data.csv")

df = df.drop(columns=["encounter_id", "patient_nbr"], errors="ignore")
df = df[df['readmitted'].isin(['NO', '<30', '>30'])]  # Keep valid readmission values
df['readmitted_binary'] = df['readmitted'].apply(lambda x: 1 if x == '<30' else 0)
df.replace('?', 'Unknown', inplace=True)

categorical_cols = df.select_dtypes(include='object').columns
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

X = df.drop(columns=['readmitted', 'readmitted_binary'])
y = df['readmitted_binary']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 🔧 Step 4: Scale + Tensor Conversion
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

class MLP(nn.Module):
    def __init__(self, input_dim):
        super(MLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

model = MLP(input_dim=X_train_tensor.shape[1])
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(300):  
    model.train()
    optimizer.zero_grad()
    output = model(X_train_tensor)
    loss = criterion(output, y_train_tensor)
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    y_pred_proba = model(X_test_tensor).numpy().flatten()
    y_pred = (y_pred_proba >= 0.5).astype(int)

sensitive_feature = X_test['race']

metric_frame = MetricFrame(
    metrics={"accuracy": accuracy_score},
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sensitive_feature
)

dp_diff = demographic_parity_difference(y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_feature)
eo_diff = equal_opportunity_difference(y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_feature)
eod_diff = equalized_odds_difference(y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_feature)

disparate_impact = MetricFrame(
    metrics=selection_rate,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sensitive_feature
).ratio()

print("🎯 Accuracy by Race Group:\n", metric_frame.by_group)
print("\n📏 Demographic Parity Difference:", dp_diff)
print("📏 Equal Opportunity Difference:", eo_diff)
print("📏 Equalized Odds Difference:", eod_diff)
print("📏 Disparate Impact Ratio:", disparate_impact)


🎯 Accuracy by Race Group:
       accuracy
race          
0     0.888257
1     0.918699
2     0.887276
3     0.876238
4     0.927536
5     0.909091

📏 Demographic Parity Difference: 0.00019707022268935164
📏 Equal Opportunity Difference: 0.0011648223645894002
📏 Equalized Odds Difference: 0.0011648223645894002
📏 Disparate Impact Ratio: 0.0


We evaluated and compared the fairness of two models — a Decision Tree and a Multilayer Perceptron (MLP) — on the diabetic dataset using race as the sensitive attribute. Our goal was to see whether these models treat all racial groups fairly when predicting whether a patient would be readmitted within 30 days.

1. Demographic Parity Difference:
The Decision Tree had a demographic parity difference of approximately 0.008, while the MLP showed an even lower value close to 0.0. This metric checks if all racial groups are predicted as "readmitted" at roughly the same rate. Both models performed well here, meaning they are not unfairly favoring one group over another in terms of how often they assign positive predictions. However, the MLP is slightly more consistent in this aspect, showing nearly perfect parity.

2. Equal Opportunity Difference:
The Decision Tree showed a small equal opportunity gap of ~0.011, while the MLP again achieved a value very close to 0. This metric checks whether the model is equally good at correctly identifying actual readmissions across races. The results suggest that both models are performing fairly well, but the MLP is slightly better in ensuring equal access to true positive outcomes, i.e., giving every group a fair chance of being correctly flagged when they are actually readmitted.

3. Equalized Odds Difference:
This metric combines fairness in both true positives and false positives. The Decision Tree had a value around 0.011, and the MLP again had a near-zero difference. This tells us that both models are making similar kinds of mistakes and correct predictions across races, which is very desirable. However, the MLP again shows slightly more uniform behavior, indicating a more balanced error pattern between groups.

4. Disparate Impact Ratio:
Here, both models produced a value of 0.0, which is a red flag. A value of 0 means that at least one race group received no positive predictions at all — which may either be due to model conservatism (not predicting any 1s) or strong class imbalance. This could indicate that while the models appear fair in terms of distribution, they may not actually be predicting any readmissions, which defeats the purpose of fairness. It's like being equally inaccurate for everyone.



In [4]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
import torch.optim as optim
from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    equal_opportunity_difference,
    equalized_odds_difference,
    selection_rate
)

df = pd.read_csv("hospital_readmissions.csv")

df['readmitted'] = df['readmitted'].map({'NO': 0, 'YES': 1, 'No': 0, 'Yes': 1, 'no': 0, 'yes': 1})
age_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35,
    '[40-50)': 45, '[50-60)': 55, '[60-70)': 65,
    '[70-80)': 75, '[80-90)': 85, '[90-100)': 95
}
df['age'] = df['age'].map(age_map)

categorical_cols = df.select_dtypes(include='object').columns
for col in categorical_cols:
    df[col] = df[col].astype(str).replace('?', 'Unknown')
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

X = df.drop(columns=['readmitted'])
y = df['readmitted']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

class MLP(nn.Module):
    def __init__(self, input_dim):
        super(MLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

model = MLP(input_dim=X_train_tensor.shape[1])
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(300):  
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    preds = model(X_test_tensor).numpy().flatten()
    y_pred = (preds >= 0.5).astype(int)

sensitive_feature = X_test['age']  # original numeric age group

metric_frame = MetricFrame(
    metrics={"accuracy": accuracy_score},
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sensitive_feature
)

dp_diff = demographic_parity_difference(y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_feature)
eo_diff = equal_opportunity_difference(y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_feature)
eod_diff = equalized_odds_difference(y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_feature)
disparate_impact = MetricFrame(
    metrics=selection_rate,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sensitive_feature
).ratio()

print("🎯 Accuracy by Age Group:\n", metric_frame.by_group)
print("\n📏 Demographic Parity Difference:", dp_diff)
print("📏 Equal Opportunity Difference:", eo_diff)
print("📏 Equalized Odds Difference:", eod_diff)
print("📏 Disparate Impact Ratio:", disparate_impact)


🎯 Accuracy by Age Group:
      accuracy
age          
45   0.663900
55   0.634385
65   0.628738
75   0.590299
85   0.564390
95   0.549383

📏 Demographic Parity Difference: 0.1795999059903502
📏 Equal Opportunity Difference: 0.21624434830693162
📏 Equalized Odds Difference: 0.21624434830693162
📏 Disparate Impact Ratio: 0.632149733831691


In [6]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    equal_opportunity_difference,
    equalized_odds_difference,
    selection_rate
)

# 📂 Step 1: Load Diabetic Dataset
df = pd.read_csv("diabetic_data.csv")

# 🧹 Step 2: Clean and Preprocess
df = df.drop(columns=["encounter_id", "patient_nbr"], errors="ignore")
df = df[df['readmitted'].isin(['NO', '<30', '>30'])]  # Valid targets only
df['readmitted_binary'] = df['readmitted'].apply(lambda x: 1 if x == '<30' else 0)  # Binary target

# Handle missing values
df = df.replace('?', 'Unknown')

# Encode all categorical variables except 'age'
categorical_cols = df.select_dtypes(include='object').columns.drop('age')
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

# 🎯 Step 3: Features and Labels
X = df.drop(columns=['readmitted', 'readmitted_binary'])
y = df['readmitted_binary']

# ⚖️ Step 4: Keep age for fairness analysis
age_sensitive = X['age']  # string ranges like '[60-70)', '[70-80)'
X = X.drop(columns=['age'])  # Drop from input so model doesn’t use it

# 🧠 Step 5: Train/Test Split
X_train, X_test, y_train, y_test, age_train, age_test = train_test_split(
    X, y, age_sensitive, test_size=0.2, stratify=y, random_state=42
)

# 📈 Step 6: Train Decision Tree
clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

# ✅ Step 7: Evaluate Fairness
metric_frame = MetricFrame(
    metrics={"accuracy": accuracy_score},
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=age_test
)

dp_diff = demographic_parity_difference(y_test, y_pred, sensitive_features=age_test)
eo_diff = equal_opportunity_difference(y_test, y_pred, sensitive_features=age_test)
eod_diff = equalized_odds_difference(y_test, y_pred, sensitive_features=age_test)
disparate_impact = MetricFrame(
    metrics=selection_rate,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=age_test
).ratio()

# 📊 Step 8: Output
print("🎯 Accuracy by Age Group:\n", metric_frame.by_group)
print("\n📏 Demographic Parity Difference:", dp_diff)
print("📏 Equal Opportunity Difference:", eo_diff)
print("📏 Equalized Odds Difference:", eod_diff)
print("📏 Disparate Impact Ratio:", disparate_impact)


🎯 Accuracy by Age Group:
           accuracy
age               
[0-10)    1.000000
[10-20)   0.946154
[20-30)   0.879630
[30-40)   0.888276
[40-50)   0.901725
[50-60)   0.913798
[60-70)   0.882120
[70-80)   0.880780
[80-90)   0.875806
[90-100)  0.869792

📏 Demographic Parity Difference: 0.05555555555555555
📏 Equal Opportunity Difference: 0.2558139534883721
📏 Equalized Odds Difference: 0.2558139534883721
📏 Disparate Impact Ratio: 0.0
